In [1]:
import pandas as pd
import datetime as dt
from dataretrieval import nwis
from dataretrieval import waterdata
from Python_Files import data
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
upstream_id = "10155200"
downstream_id = "10155500"
start_date = '2016-01-01'
end_date = '2025-12-31'

discharge = data.get_discharge(upstream_id,downstream_id,start_date,end_date)
#width = data.get_width()

In [3]:
# possible_validation = pd.merge(discharge,cleaning,how='inner',on='Date').dropna()
# sample_daily["Downstream Width (ft)"] = sample_daily["Downstream Width (ft)"].ffill().bfill()
# possible_validation.head(30)

In [4]:
datum, up_MSE, down_MSE = data.log_transform(discharge,'depth')
print(f"Upstream Gauge Depth Training MSE: {up_MSE:.4f}")
print(f"Downstream Gauge Depth Training MSE: {down_MSE:.4f}")

w, up_MSE, down_MSE = data.log_transform(discharge,"width")
print(f"Upstream Width Training MSE: {up_MSE:.4f}")
print(f"Downstream Width Training MSE: {down_MSE:.4f}")

w = w.drop(columns=["Downstream Mean Discharge (cfs)","Upstream Mean Discharge (cfs)"],axis=1)


Upstream Gauge Depth Training MSE: 0.0018
Downstream Gauge Depth Training MSE: 0.0007
Upstream Width Training MSE: 0.0055
Downstream Width Training MSE: 0.0041


In [5]:
years = np.array(datum.index.year.unique())
years = years.reshape((len(years)//2),2)
surface = data.get_surface(upstream_id,downstream_id,years)

In [6]:
#merge and add the to surface and to gauge values

total = pd.merge(pd.merge(surface,datum,how='outer',on="Date"),w,how='outer',on='Date')
total["Upstream Depth (ft)"] = total["Upstream Datum (ft)"] + total["Upstream Depth (ft)"]
total["Downstream Depth (ft)"] = total["Downstream Datum (ft)"] + total["Downstream Depth (ft)"]
total = total.drop(columns=["Upstream Datum (ft)","Downstream Datum (ft)"])
total.to_csv('../Data Files/Raw/Raw_Data.csv')

In [7]:
data = total.dropna()
range = pd.date_range(start=data.index.min(), end=total.index.max(), freq='D')
total.to_csv('../Data Files/Cleaned/Data.csv')

missing_dates = range.difference(data.index)
print("Missing dates:", missing_dates)

Missing dates: DatetimeIndex(['2017-05-11', '2017-05-12', '2017-05-13', '2017-05-14',
               '2017-05-15', '2017-05-16', '2017-05-17', '2017-08-15',
               '2017-08-16', '2017-08-17', '2017-08-18', '2017-08-19',
               '2017-08-20', '2019-06-10', '2019-06-11', '2019-06-12',
               '2019-06-13', '2019-06-14', '2019-06-15', '2019-06-16',
               '2019-06-17', '2019-06-18', '2019-06-19', '2019-06-20',
               '2019-06-21', '2019-06-22', '2019-06-23', '2019-08-15',
               '2019-08-16', '2019-08-17', '2019-08-18', '2019-08-19',
               '2019-08-20', '2019-08-21', '2023-01-21', '2023-01-22',
               '2023-01-23', '2023-01-24', '2023-05-04', '2023-07-26',
               '2023-07-27', '2023-07-28', '2023-07-29', '2023-07-30',
               '2024-08-03', '2024-08-04', '2024-08-05', '2024-08-06',
               '2024-08-08', '2024-08-09', '2024-08-10', '2024-08-11',
               '2024-08-12', '2024-08-13', '2024-08-14', '2024